# MBPS Environment Setup — Anydesk A6000 Pro (48GB)

Complete setup for the CUPS unsupervised panoptic segmentation pipeline.

**Machine**: Anydesk RTX A6000 Pro (48GB VRAM), conda env `ups`  
**Pipeline**: CUPS Cascade Mask R-CNN + DINOv3 ViT-B/16 backbone (frozen)  

### What gets installed from source:
- **PyTorch + TorchVision** (CUDA 12.1 wheels)
- **Detectron2** (built from source against local CUDA + PyTorch)
- **DINOv3** (Meta's official repo, editable install — requires Python >= 3.11)
- **CUPS** (added to PYTHONPATH — no setup.py, not pip-installable)
- **MBPS** (editable install from project root)

### Prerequisites:
- Ubuntu 20.04+ with NVIDIA drivers installed
- CUDA Toolkit 12.1+ installed (`nvcc --version` should work)
- conda or miniconda installed
- `git` available
- Project cloned to the machine (with `refs/cups/`, `refs/dinov3/`, `weights/`)

---
## 0. Configuration

All machine-specific paths in one place. Edit these before running anything.

In [ ]:
# ============================================================
# EDIT THESE PATHS FOR YOUR MACHINE
# ============================================================

# Conda environment name
ENV_NAME = "ups"

# Project root on this machine
PROJECT_ROOT = "/home/santosh/mbps_panoptic_segmentation"

# Cityscapes dataset root
DATA_ROOT = "/media/santosh/Kuldeep/panoptic_segmentation/datasets/cityscapes"

# Experiment output directory
LOG_PATH = "/media/santosh/Kuldeep/panoptic_segmentation/experiments"

# ============================================================
# Derived paths (do not edit)
# ============================================================
import os

CUPS_ROOT = os.path.join(PROJECT_ROOT, "refs", "cups")
DINOV3_ROOT = os.path.join(PROJECT_ROOT, "refs", "dinov3")
WEIGHTS_DIR = os.path.join(PROJECT_ROOT, "weights")

print(f"Project root:  {PROJECT_ROOT}")
print(f"CUPS root:     {CUPS_ROOT}")
print(f"DINOv3 root:   {DINOV3_ROOT}")
print(f"Weights dir:   {WEIGHTS_DIR}")
print(f"Data root:     {DATA_ROOT}")
print(f"Log path:      {LOG_PATH}")

---
## 1. Pre-flight Checks

Verify the machine has GPU, CUDA toolkit, and conda.

In [ ]:
%%bash
echo "=== GPU ==="
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
echo ""
echo "=== CUDA Toolkit ==="
nvcc --version 2>/dev/null || echo "ERROR: nvcc not found. Install CUDA Toolkit first."
echo ""
echo "=== GCC ==="
gcc --version | head -1
echo ""
echo "=== Python ==="
python3 --version
echo ""
echo "=== Conda ==="
conda --version 2>/dev/null || echo "WARNING: conda not found."

---
## 2. Create Conda Environment

**Python 3.11** is required because DINOv3 (Meta's official repo) specifies `python_requires >= 3.11`.

In [ ]:
%%bash
conda create -n ups python=3.11 -y
echo "Done. Environment 'ups' created with Python 3.11."

**IMPORTANT**: After creating the environment, activate it in your terminal:
```bash
conda activate ups
```
Then re-launch Jupyter from within the activated environment:
```bash
pip install jupyterlab ipykernel
python -m ipykernel install --user --name ups --display-name "UPS (Python 3.11)"
jupyter lab
```
Select the **UPS (Python 3.11)** kernel before continuing.

In [ ]:
import sys
print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")

# Verify Python >= 3.11
assert sys.version_info >= (3, 11), (
    f"Python >= 3.11 required (DINOv3 dependency). Got {sys.version_info.major}.{sys.version_info.minor}"
)

# Verify conda env name
assert "ups" in sys.executable or "ups" in sys.prefix, (
    f"Not in 'ups' conda env! Current: {sys.prefix}\n"
    "Run: conda activate ups"
)
print("Environment OK.")

---
## 3. Set Environment Variables

Configure CUDA paths and build flags for source compilations.

In [ ]:
import os
import subprocess

# Auto-detect CUDA home
cuda_candidates = [
    os.environ.get("CUDA_HOME", ""),
    "/usr/local/cuda",
    "/usr/local/cuda-12.1",
    "/usr/local/cuda-12.4",
    "/usr/local/cuda-11.8",
]

CUDA_HOME = None
for c in cuda_candidates:
    if c and os.path.isfile(os.path.join(c, "bin", "nvcc")):
        CUDA_HOME = c
        break

if CUDA_HOME is None:
    raise RuntimeError(
        "CUDA_HOME not found. Install CUDA Toolkit or set CUDA_HOME manually.\n"
        "  e.g.: export CUDA_HOME=/usr/local/cuda-12.1"
    )

os.environ["CUDA_HOME"] = CUDA_HOME
os.environ["PATH"] = f"{CUDA_HOME}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = (
    f"{CUDA_HOME}/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
)

# A6000 Pro compute capability (Ampere=8.6, Ada=8.9)
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.6;8.9"

# Speed up compilation
os.environ["MAX_JOBS"] = "8"

# Verify
nvcc_out = subprocess.check_output([f"{CUDA_HOME}/bin/nvcc", "--version"], text=True)
print(f"CUDA_HOME: {CUDA_HOME}")
print(nvcc_out.strip().split("\n")[-1])

---
## 4. Install PyTorch + TorchVision (CUDA 12.1 wheels)

Using pre-built wheels from PyTorch's official index.

In [ ]:
%%bash
# Install PyTorch with CUDA 12.1 support
# If your CUDA toolkit is 11.8, change cu121 -> cu118
pip install torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    x = torch.randn(2, 3, device="cuda")
    print(f"Tensor on GPU: {x.device} -- OK")
else:
    raise RuntimeError("CUDA not available! Check PyTorch installation.")

---
## 5. Install Detectron2 from Source

Detectron2 must be built against your local PyTorch + CUDA.

In [ ]:
%%bash
pip install ninja cython

DETECTRON2_DIR="/tmp/detectron2_build"
if [ ! -d "$DETECTRON2_DIR" ]; then
    git clone https://github.com/facebookresearch/detectron2.git "$DETECTRON2_DIR"
fi
cd "$DETECTRON2_DIR"
git pull

pip install -e . 2>&1 | tail -5
echo ""
echo "Detectron2 installed."

In [ ]:
import detectron2
from detectron2.config import get_cfg
print(f"Detectron2: {detectron2.__version__}")
print("Detectron2 import OK")

---
## 6. Install DINOv3 from Source

Official Meta DINOv3 repo. Requires Python >= 3.11.  
Installed as editable package (`pip install -e .`) since it has a proper `setup.py`.

In [ ]:
%%bash
# DINOv3 dependencies
pip install ftfy omegaconf regex scikit-learn submitit termcolor torchmetrics

# Install DINOv3 from source (editable)
cd "$HOME/mbps_panoptic_segmentation/refs/dinov3"
pip install -e . 2>&1 | tail -5
echo ""
echo "DINOv3 installed."

In [ ]:
import dinov3
print(f"DINOv3: {dinov3.__version__}")

# Verify the hub backbone function exists
from dinov3.hub.backbones import dinov3_vitb16
print("dinov3.hub.backbones.dinov3_vitb16 -- OK")

---
## 7. Install CUPS Dependencies + PYTHONPATH

CUPS does **not** have a `setup.py` — it cannot be pip-installed.  
Instead, we install its pip dependencies and add it to `PYTHONPATH`.  
This matches how the codebase loads CUPS internally (via `sys.path.insert` in `backbone_dinov3_vit.py`).

In [ ]:
%%bash
pip install \
    "kornia>=0.6.11" \
    yacs \
    "pytorch-lightning>=2.0.0" \
    "torchmetrics==1.3.1" \
    "timm==0.9.12" \
    pycocotools \
    "scikit-learn==1.3.2" \
    "scikit-image>=0.22.0" \
    "rtpt==0.0.4" \
    colored \
    faster-coco-eval \
    fvcore \
    "wandb>=0.15.4" \
    einops \
    "av>=11.0.0" \
    optuna

echo "CUPS pip dependencies installed."

In [ ]:
import sys
import os

# Add CUPS to Python path (same as backbone_dinov3_vit.py does internally)
CUPS_ROOT = os.path.expanduser("~/mbps_panoptic_segmentation/refs/cups")
if CUPS_ROOT not in sys.path:
    sys.path.insert(0, CUPS_ROOT)

# Verify CUPS imports
import cups
from cups.config import get_cfg_defaults
print(f"CUPS imported from: {cups.__file__}")
print("cups.config.get_cfg_defaults -- OK")

---
## 8. Install MBPS Project (Editable)

The main project package (has `setup.py`).

In [ ]:
%%bash
cd "$HOME/mbps_panoptic_segmentation"
pip install -e . 2>&1 | tail -5
echo ""
echo "MBPS installed."

---
## 9. Standard pip Packages

Everything else that doesn't need source compilation.

In [ ]:
%%bash
pip install \
    "numpy>=1.24.0" \
    "scipy>=1.11.0" \
    "Pillow>=10.0.0" \
    "opencv-python-headless>=4.8.0" \
    "matplotlib>=3.7.0" \
    "pandas>=2.0.0" \
    tqdm \
    "PyYAML>=6.0" \
    "omegaconf>=2.3.0" \
    "hydra-core>=1.3.0" \
    "tensorboard>=2.15.0" \
    panopticapi \
    cityscapesscripts \
    pydensecrf \
    "networkx>=3.1" \
    h5py \
    ipython \
    "absl-py>=2.0.0" \
    "ml_collections>=0.1.1" \
    ftfy \
    regex \
    termcolor \
    "shapely>=1.8.2" \
    huggingface-hub

echo "Standard packages installed."

---
## 10. Optional: pycocotools from Source

If `import pycocotools` fails, build from source.

In [ ]:
%%bash
# Uncomment if pycocotools pip install failed:
# pip install cython
# pip install "git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI"

---
## 11. Set PYTHONPATH + LD_LIBRARY_PATH Permanently

Write conda activation hooks so CUPS, DINOv3, and shared libraries are always on the path.

In [ ]:
%%bash
PROJECT="$HOME/mbps_panoptic_segmentation"

ACTIVATE_DIR="$CONDA_PREFIX/etc/conda/activate.d"
DEACTIVATE_DIR="$CONDA_PREFIX/etc/conda/deactivate.d"
mkdir -p "$ACTIVATE_DIR" "$DEACTIVATE_DIR"

cat > "$ACTIVATE_DIR/mbps_env.sh" << EOF
#!/bin/bash
# --- LD_LIBRARY_PATH ---
export _OLD_LD_LIBRARY_PATH="\$LD_LIBRARY_PATH"
export LD_LIBRARY_PATH="\$CONDA_PREFIX/lib:\$LD_LIBRARY_PATH"

# --- PYTHONPATH: CUPS (no setup.py) + project root ---
export _OLD_PYTHONPATH="\$PYTHONPATH"
export PYTHONPATH="${PROJECT}/refs/cups:${PROJECT}:\$PYTHONPATH"
EOF

cat > "$DEACTIVATE_DIR/mbps_env.sh" << 'EOF'
#!/bin/bash
export LD_LIBRARY_PATH="$_OLD_LD_LIBRARY_PATH"
unset _OLD_LD_LIBRARY_PATH
export PYTHONPATH="$_OLD_PYTHONPATH"
unset _OLD_PYTHONPATH
EOF

echo "Conda activation hooks installed:"
echo "  $ACTIVATE_DIR/mbps_env.sh"
echo "  $DEACTIVATE_DIR/mbps_env.sh"
echo ""
echo "After 'conda activate ups', PYTHONPATH will include:"
echo "  ${PROJECT}/refs/cups"
echo "  ${PROJECT}"

---
## 12. Download DINOv3 Weights

Downloads weights using the official DINOv3 API (Meta CDN), then saves locally to `weights/dinov3_vitb16_official.pth` for fast loading.

In [ ]:
import os
import torch

WEIGHTS_DIR = os.path.expanduser("~/mbps_panoptic_segmentation/weights")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

weights_path = os.path.join(WEIGHTS_DIR, "dinov3_vitb16_official.pth")

if os.path.exists(weights_path):
    size_mb = os.path.getsize(weights_path) / 1e6
    print(f"Weights already exist: {weights_path} ({size_mb:.1f} MB)")
else:
    print("Downloading DINOv3 ViT-B/16 from Meta CDN...")
    from dinov3.hub.backbones import dinov3_vitb16

    # pretrained=True triggers download from Meta's official URL
    model = dinov3_vitb16(pretrained=True)
    torch.save(model.state_dict(), weights_path)

    size_mb = os.path.getsize(weights_path) / 1e6
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Saved: {weights_path} ({size_mb:.1f} MB, {n_params/1e6:.1f}M params)")
    del model

---
## 13. Full Verification

Import every critical package and confirm versions.

In [ ]:
import sys
import os
import importlib

# Ensure CUPS is on path for this session
cups_path = os.path.expanduser("~/mbps_panoptic_segmentation/refs/cups")
if cups_path not in sys.path:
    sys.path.insert(0, cups_path)

REQUIRED = {
    # === Source-built ===
    "torch": "PyTorch",
    "torchvision": "TorchVision",
    "detectron2": "Detectron2",
    "dinov3": "DINOv3",
    # === CUPS pipeline (via PYTHONPATH) ===
    "cups": "CUPS",
    "kornia": "Kornia",
    "pytorch_lightning": "PyTorch Lightning",
    "torchmetrics": "TorchMetrics",
    "timm": "timm",
    "fvcore": "fvcore",
    "yacs": "yacs",
    "pycocotools": "pycocotools",
    # === Vision / Data ===
    "cv2": "OpenCV",
    "PIL": "Pillow",
    "skimage": "scikit-image",
    "sklearn": "scikit-learn",
    "scipy": "SciPy",
    "numpy": "NumPy",
    # === Config / Logging ===
    "yaml": "PyYAML",
    "omegaconf": "OmegaConf",
    "wandb": "W&B",
    "tensorboard": "TensorBoard",
    # === Utilities ===
    "einops": "einops",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "panopticapi": "panopticapi",
}

print(f"{'Package':<25} {'Status':<10} {'Version'}")
print("-" * 55)

failures = []
for module, name in REQUIRED.items():
    try:
        mod = importlib.import_module(module)
        ver = getattr(mod, "__version__", "--")
        print(f"{name:<25} {'OK':<10} {ver}")
    except ImportError as e:
        print(f"{name:<25} {'FAIL':<10} {e}")
        failures.append(name)

print("\n" + "=" * 55)
if failures:
    print(f"FAILED: {', '.join(failures)}")
    print("Fix these before proceeding.")
else:
    print("All packages OK.")

In [ ]:
# GPU smoke test
import torch
import torch.nn as nn

device = torch.device("cuda:0")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print()

# 1. Basic CUDA matmul
a = torch.randn(256, 256, device=device)
c = a @ a.T
print(f"[1/4] CUDA matmul: {c.shape} -- OK")

# 2. Detectron2 backbone
from detectron2.config import get_cfg
from detectron2.modeling import build_backbone
cfg = get_cfg()
cfg.MODEL.BACKBONE.NAME = "build_resnet_backbone"
cfg.MODEL.RESNETS.DEPTH = 50
cfg.MODEL.RESNETS.OUT_FEATURES = ["res4"]
cfg.MODEL.RESNETS.NORM = "BN"
cfg.INPUT.FORMAT = "RGB"
backbone = build_backbone(cfg).to(device)
with torch.no_grad():
    out = backbone(torch.randn(1, 3, 224, 224, device=device))
print(f"[2/4] Detectron2 ResNet-50: {list(out.keys())} -- OK")
del backbone, out
torch.cuda.empty_cache()

# 3. timm ViT
import timm
vit = timm.create_model("vit_small_patch16_224", pretrained=False).to(device)
with torch.no_grad():
    out = vit(torch.randn(1, 3, 224, 224, device=device))
print(f"[3/4] timm ViT-S/16: {out.shape} -- OK")
del vit, out
torch.cuda.empty_cache()

# 4. AMP autocast
with torch.cuda.amp.autocast():
    y = nn.Linear(768, 256).to(device)(torch.randn(64, 768, device=device))
print(f"[4/4] AMP autocast: {y.dtype} -- OK")
del y
torch.cuda.empty_cache()

print("\nAll GPU smoke tests passed.")

---
## 14. Verify CUPS End-to-End

Load CUPS config, verify the DINOv3 backbone builds, and check data paths.

In [ ]:
import sys
import os

cups_path = os.path.expanduser("~/mbps_panoptic_segmentation/refs/cups")
if cups_path not in sys.path:
    sys.path.insert(0, cups_path)

import cups

# Load the Anydesk training config
config_path = os.path.join(
    cups_path, "configs", "train_cityscapes_dinov3_vitb_k80_anydesk.yaml"
)

if os.path.exists(config_path):
    cfg = cups.get_default_config(experiment_config_file=config_path)
    print(f"Config loaded: {config_path}")
    print(f"  Backbone:    {cfg.MODEL.BACKBONE_TYPE}")
    print(f"  Frozen:      {cfg.MODEL.DINOV2_FREEZE}")
    print(f"  Batch size:  {cfg.TRAINING.BATCH_SIZE} x {cfg.TRAINING.ACCUMULATE_GRAD_BATCHES} accum")
    print(f"  Steps:       {cfg.TRAINING.STEPS}")
    print(f"  Precision:   {cfg.TRAINING.PRECISION}")
    print(f"  Data root:   {cfg.DATA.ROOT}")
    print(f"  Pseudo root: {cfg.DATA.ROOT_PSEUDO}")
else:
    # Fallback: just load default config
    cfg = cups.get_default_config()
    print(f"Default config loaded (Anydesk config not found at {config_path})")
    print(f"  Backbone: {cfg.MODEL.BACKBONE_TYPE}")

In [ ]:
# Verify data paths exist on this machine
import os

data_root = "/media/santosh/Kuldeep/panoptic_segmentation/datasets/cityscapes"

checks = {
    "leftImg8bit/train": "Training images",
    "leftImg8bit/val": "Validation images",
    "gtFine/val": "Ground truth (val)",
}

print(f"Data root: {data_root}")
print(f"Exists: {os.path.isdir(data_root)}")
print()

for subdir, desc in checks.items():
    path = os.path.join(data_root, subdir)
    exists = os.path.isdir(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {desc}: {subdir}")

# Check pseudo-labels
pseudo_root = os.path.join(data_root, "cups_pseudo_labels_dinov3_k80")
if os.path.isdir(pseudo_root):
    n_train = len(os.listdir(os.path.join(pseudo_root, "train"))) if os.path.isdir(os.path.join(pseudo_root, "train")) else 0
    n_val = len(os.listdir(os.path.join(pseudo_root, "val"))) if os.path.isdir(os.path.join(pseudo_root, "val")) else 0
    print(f"  [OK] Pseudo-labels: {n_train} train, {n_val} val")
else:
    print(f"  [MISSING] Pseudo-labels not found at {pseudo_root}")
    print("           Generate with: python mbps_pytorch/generate_semantic_pseudolabels_dinov3.py")

---
## 15. Print Final Environment Summary

In [ ]:
import torch
import platform
import sys

print("=" * 60)
print("MBPS ENVIRONMENT SUMMARY")
print("=" * 60)
print(f"OS:             {platform.platform()}")
print(f"Python:         {sys.version.split()[0]}")
print(f"Conda env:      {os.path.basename(sys.prefix)}")
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA (torch):   {torch.version.cuda}")
print(f"cuDNN:          {torch.backends.cudnn.version()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"GPU count:      {torch.cuda.device_count()}")

import detectron2
print(f"Detectron2:     {detectron2.__version__}")

import dinov3
print(f"DINOv3:         {dinov3.__version__}")

import timm
print(f"timm:           {timm.__version__}")

print(f"CUPS:           via PYTHONPATH")
print("=" * 60)
print("Environment ready.")

---
## 16. Save Environment Snapshot

In [ ]:
%%bash
SNAPSHOT_DIR="$HOME/mbps_panoptic_segmentation/env_snapshots"
mkdir -p "$SNAPSHOT_DIR"

TIMESTAMP=$(date +%Y%m%d_%H%M%S)
pip freeze > "$SNAPSHOT_DIR/requirements_frozen_${TIMESTAMP}.txt"
conda list > "$SNAPSHOT_DIR/conda_list_${TIMESTAMP}.txt" 2>/dev/null
nvidia-smi > "$SNAPSHOT_DIR/gpu_info_${TIMESTAMP}.txt"

echo "Environment snapshot saved to $SNAPSHOT_DIR/"
ls -la "$SNAPSHOT_DIR/"*${TIMESTAMP}*

---
## 17. Training Dry Run

Verify the CUPS training script loads without errors.

In [ ]:
%%bash
cd "$HOME/mbps_panoptic_segmentation/refs/cups"
python train.py --help 2>&1 | head -20
echo ""
echo "train.py --help OK"

---
## Quick Reference: Running Training

```bash
conda activate ups

# === Stage-2: Train Cascade Mask R-CNN on pseudo-labels ===
cd ~/mbps_panoptic_segmentation/refs/cups
nohup python -u train.py \
    --experiment_config_file configs/train_cityscapes_dinov3_vitb_k80_anydesk.yaml \
    > ~/logs/cups_stage2.log 2>&1 &
echo $!
tail -f ~/logs/cups_stage2.log

# === Stage-3: Self-training (after Stage-2 completes) ===
cd ~/mbps_panoptic_segmentation/refs/cups
nohup python -u train_self.py \
    --experiment_config_file configs/train_self_cityscapes_dinov3_vitb_k80_2gpu.yaml \
    --ckpt_path /path/to/stage2_best.ckpt \
    > ~/logs/cups_stage3.log 2>&1 &
echo $!
tail -f ~/logs/cups_stage3.log

# === Conv-DoRA experiment ===
cd ~/mbps_panoptic_segmentation/refs/cups
nohup python -u train.py \
    --experiment_config_file configs/train_cityscapes_dinov3_vitb_k80_conv_dora.yaml \
    > ~/logs/cups_conv_dora.log 2>&1 &
echo $!
tail -f ~/logs/cups_conv_dora.log
```